# Lab 5 — 把 Lab 1–4 翻成 **真的 ADK 2.0 graph**

> 這份 notebook 用的是**正式版 `google-adk 2.x`**（Python 2.0 已於 2026-05 GA），
> 底下的 graph 是**真的跑起來**的，不是示意圖。

**這一關的一句話**：流程從「LLM 的即興行為」→「程式的一個資料結構」。

對照你在 Lab 1–4 親手踩過的三個「1.x 的勉強」：

| 你在哪一關踩到 | 1.x 的做法（勉強） | 2.0 的解法 |
|---|---|---|
| Lab 2 兩個研究員 | `SequentialAgent` 只能排隊 | 從 `START` 拉兩條邊 = **天然並行** |
| Lab 2 迴圈跳出 | 靠 reviewer **記得**呼叫 approve（拜託 LLM） | `route` 是**確定的 if**，同 state 走同邊 |
| Lab 3 護欄/記憶 | callback 要**掛對 agent**（掛錯默默失效） | 節點/邊上**看得見位置** |

## 0. 安裝正式版 ADK 2.0（Colab 雲端 VM，隔離、零風險）

In [ ]:
!pip install -q "google-adk>=2.0"
import importlib.metadata as m
print("google-adk", m.version("google-adk"), "→ 這是正式版 2.x，不是 alpha")

## 1. 差別①：序列 vs 並行（先看原理，數字看得見）

用兩個「假研究員」（`sleep` 模擬等 LLM/等工具的時間）示範：1.x 排隊 vs 2.0 並行。
那 1.2 秒代表真實世界「等回應」的時間——並行就是把兩邊的等待**疊在一起**。

In [ ]:
import asyncio, time   # asyncio＝非同步（號碼牌制）工具箱；time＝碼錶

# 「假研究員」：睡 1.2 秒，模擬「等 LLM 回覆／等工具查詢」的等待時間。
# async def＝這個函式支援「號碼牌制」：等待時可以把 CPU 讓給別人做事
async def research(name, seconds=1.2):
    await asyncio.sleep(seconds)   # await＝拿號碼牌等叫號（等待中，別人照常工作）
    return f"{name}_result"

async def sequential():   # 1.x SequentialAgent 的行為：排隊——一個做完才換下一個
    t = time.perf_counter()                                # 按下碼錶
    await research("trend"); await research("audience")   # 先等 A 整整 1.2s，再等 B 再 1.2s
    return time.perf_counter() - t                         # 總耗時 ≈ 2.4s

async def parallel():     # 2.0 graph 的行為：兩條邊從 START 出發＝同時開工
    t = time.perf_counter()
    await asyncio.gather(research("trend"), research("audience"))
    # ↑ gather＝同時發兩張號碼牌、等「全部」完成——兩段 1.2s 的等待疊在一起
    return time.perf_counter() - t                         # 總耗時 ≈ 1.2s（不是 2.4！）

seq = await sequential()   # notebook 可以直接寫 await（Colab 自帶事件迴圈）
par = await parallel()
print(f"1.x 序列：{seq:.2f}s")
print(f"2.0 並行：{par:.2f}s")
print(f"→ 並行快了約 {seq-par:.2f}s（≈ {seq/par:.1f}x）；研究員越多、差距越大")

## 2. 差別②：**真的用 ADK 2.0 graph 跑一次**

同一套「保健品文案小組」，用 2.0 的 `Workflow / Edge / FunctionNode / JoinNode / START`：

```
START ─┬─> trend_research ────┐
       └─> audience_research ─┴─> [Join] ─> combine
```

- 兩條邊同時從 `START` 出發 = **並行**（取代 Lab 2 排隊）
- `JoinNode` 等兩邊都到齊，才進 `combine`
- 節點就是「工作」、邊就是「流程」——流程變成一個**資料結構**

In [ ]:
# ── 匯入 2.0 的圖形元件 ──────────────────────────────────────
from google.adk.workflow import Workflow, Edge, FunctionNode, JoinNode, START
#    Workflow＝整張圖｜Edge＝一條邊（誰接誰）｜FunctionNode＝把普通函式包成節點
#    JoinNode＝會合點（等所有前驅到齊）｜START＝圖的起點記號
from google.adk.runners import InMemoryRunner   # 跑圖用的 runner（跟 Lab 1-4 同一位「郵差」）
from google.genai import types                  # 組訊息用的型別（Content/Part）

# ── 三個節點：都是普通 Python function，靠 ctx.state（白板）傳話 ──
# ctx＝框架呼叫你的函式時自動塞進來的「工作證」——
#      跟 Lab 2 工具裡的 tool_context 是同一家族；最常用的就是 ctx.state 白板
def trend_research(ctx) -> dict:            # -> dict＝type hint：我會回傳一個 dict
    ctx.state["trend"] = "魚油需求上升"       # 往白板寫字（state 用起來就像 dict）
    return {"trend": ctx.state["trend"]}    # return 的 dict＝交棒給下一個節點的包裹

def audience_research(ctx) -> dict:
    ctx.state["audience"] = "30-45 上班族"
    return {"audience": ctx.state["audience"]}

def combine(ctx) -> dict:
    # .get()＝安全讀白板：key 不存在回 None、不報錯（預設值管空白）
    out = f"文案：給{ctx.state.get('audience')}，主打{ctx.state.get('trend')}"
    ctx.state["copy"] = out
    return {"copy": out}

# ── 把函式「包」成圖上的節點（name 是節點在圖上的名字）──
n_trend = FunctionNode(func=trend_research, name="trend")
n_aud   = FunctionNode(func=audience_research, name="audience")
n_join  = JoinNode(name="join")             # 會合點不需要函式：它的工作就是「等人到齊」
n_comb  = FunctionNode(func=combine, name="combine")

# ── 圖：整個流程是「一個資料結構」（節點＋邊），不是巢狀包裝盒 ──
#
#         START
#         ╱    ╲        ← 兩條邊從 START 出發 ＝ 天然並行（1.x 要 ParallelAgent 包）
#    trend      audience
#         ╲    ╱
#          join          ← 等兩位研究員「都」交作業
#           │
#        combine         ← 會合後才組文案
#
wf = Workflow(
    name="supplement_flow",
    edges=[
        Edge(from_node=START,   to_node=n_trend),   # ┐ 兩條邊從 START
        Edge(from_node=START,   to_node=n_aud),     # ┘ ＝ 同時跑
        Edge(from_node=n_trend, to_node=n_join),    # ┐ 兩條匯入會合點
        Edge(from_node=n_aud,   to_node=n_join),    # ┘
        Edge(from_node=n_join,  to_node=n_comb),    #   到齊才前進
    ],
)
print("Workflow 建好了：", wf.name)
# 讀懂流程只要讀 edges 這個 list——不用讀 prompt。這就是「流程即資料結構」。

In [ ]:
# ── 用 InMemoryRunner 跑（跟 Lab 1-4 一模一樣的跑法：圖也是一種 agent）──
async def run_graph():
    runner = InMemoryRunner(agent=wf, app_name="lab5")
    #        ↑ 郵差！直接把「圖」當 agent 交給他——2.0 裡圖和 agent 是同一等公民

    sess = await runner.session_service.create_session(app_name="lab5", user_id="u")
    #      ↑ 開一本新筆記本（session）：白板 state 就掛在它身上

    async for ev in runner.run_async(                 # run_async 會「一顆一顆」吐事件
        user_id="u", session_id=sess.id,
        new_message=types.Content(role="user", parts=[types.Part(text="go")]),
    ):                                                # ↑ 使用者訊息（圖的啟動訊號）
        pass   # 事件流＝郵差的工作日誌；這個 demo 不逐顆看，只驗收最後的白板

    # 重新取回 session，讀最終 state——三個節點寫上白板的東西都在這
    s = await runner.session_service.get_session(app_name="lab5", user_id="u", session_id=sess.id)
    return dict(s.state)

state = await run_graph()
print("最終 state：")
for k, v in state.items():
    print(f"  {k} = {v}")   # 預期看到 trend / audience / copy 三個 key
print("\n✅ 這是真的 ADK 2.0 graph 跑出來的——兩個研究員並行、join 彙整、再產文案。")

## 3. 收尾金句

> **1.x 讓 LLM「即興」控制流程**（靠 prompt 拜託、靠 callback 掛對位置）；
> **2.0 先「畫好」控制流程**（節點、邊、條件），再讓 LLM 去填節點裡的內容。

一句話：**流程從「LLM 的一種行為」變成「程式的一個資料結構」——可測試、可重現，還免費拿到並行。**

延伸（不做，只點到）：**MCP 是 agent 接工具，A2A 是 agent 接 agent**——那是 capstone 的主題。